# OPERA RTC-S1 — analysis-ready backscatter workflow

The OPERA Radiometric Terrain Corrected Sentinel-1 product (`opera-rtc-s1`) is NASA's analysis-ready, terrain-corrected backscatter mirror of the Sentinel-1 mission. Unlike the raw SLC stack, RTC is search-only: it's already a processed output, not an InSAR input. This walkthrough searches the OPERA RTC catalog over a bbox and inspects the available scenes.

Needs the `[asf]` extra (`pip install earthlens[asf]`).

## Why this product?

RTC is meant for **amplitude / backscatter** analyses — flood mapping, land-cover change, urban growth, forest disturbance — where the interferometric phase is not needed. The terrain correction makes the values directly comparable across acquisitions (and across satellites in the wider OPERA-S1 family) without per-pixel reprocessing.

Because the SLC has been collapsed into amplitude, **stacking is not meaningful** — the catalog flags this row as `stackable: false`, and the backend rejects stack mode with a clear error:

In [ ]:
from earthlens.asf import Catalog

row = Catalog().get_product('opera-rtc-s1')
print(f'platform={row.platform!r}  dataset={row.dataset!r}  product_type={row.product_type!r}')
print(f'stackable: {row.stackable}')

## Search by bbox + window

OPERA RTC selects via `DATASET.OPERA_S1` rather than a `platform` — the backend routes that through `geo_search(dataset='OPERA_S1', processingLevel='RTC', ...)` transparently.

In [ ]:
from earthlens.earthlens import EarthLens

el = EarthLens(
    data_source='asf',
    variables=['opera-rtc-s1'],          # alias 'opera-rtc' also works
    start='2024-06-01',
    end='2024-06-15',
    lat_lim=[37.0, 37.5],                # San Francisco Bay
    lon_lim=[-122.5, -122.0],
    path='opera_rtc_out',
    max_results=20,
)
products = el.datasource._search()
print(f'{len(products)} RTC product(s) match the request')

### Inspect what came back

Each RTC scene carries its acquisition time, the satellite (S1A / S1C), and pixel-spacing metadata. We pull a few fields for orientation.

In [ ]:
for product in products[:5]:
    props = product.metadata['product'].properties
    start = props.get('startTime', '?')
    platform = props.get('platform', '?')
    print(f'  {start}  {platform}  {product.metadata["fileName"]}')

## Companion products

Each RTC scene has a **static-layer mask** companion (`opera-rtc-s1-static`) carrying topography masks, no-data zones, and similar per-burst metadata. The mask shares the same dataset key but a different `product_type`; pull it as a separate request when needed for masking analyses.

In [ ]:
static = EarthLens(
    data_source='asf',
    variables=['opera-rtc-s1-static'],
    start='2024-06-01',
    end='2024-06-15',
    lat_lim=[37.0, 37.5],
    lon_lim=[-122.5, -122.0],
    path='opera_rtc_static_out',
    max_results=5,
)
static_products = static.datasource._search()
print(f'{len(static_products)} static-layer mask product(s)')

## Disturbance alerts: a different OPERA-S1 product

If you want **change detection** rather than raw backscatter, OPERA also publishes `opera-dist-alert-s1` — a land-surface disturbance alert layer derived from the RTC time series. It's lower-latency and analysis-ready.

In [ ]:
dist = EarthLens(
    data_source='asf',
    variables=['opera-dist-alert-s1'],
    start='2024-06-01',
    end='2024-06-15',
    lat_lim=[37.0, 37.5],
    lon_lim=[-122.5, -122.0],
    path='opera_dist_out',
    max_results=5,
)
dist_products = dist.datasource._search()
print(f'{len(dist_products)} disturbance-alert product(s)')

## Downloading

All three searches above ran anonymously. To pull the products to disk, set EDL credentials and call `el.download()` (see [Authentication](../../reference/asf/authentication.md)). The download path is shared with the InSAR stack: same `ASFAuth`, same idempotent skip-existing logic.